# Task 1 · Find Where the Image Changes (20 marks)

ES666 Assignment 3 · Lunar Landing: The Final Descent. The full assignment is
in the handout, `ES666_A3_Handout.pdf`, and in `tasks.md`.

Learn how motion is written into image intensities: compute image gradients,
connect them to motion, and find the points that can be tracked. Images are
arrays indexed `image[y, x]` (y is the row, x the column); points are `[x, y]`.
Do not use `np.gradient`.

**How this notebook works**

* Write your code only in the cells whose first line starts with `# YOUR CODE`.
  Only those cells are graded, and they are read from the **saved** notebook.
  Add your own cells anywhere for experiments or notes: they are never graded.
  A helper function you want graded with your code goes in a `# YOUR CODE` cell
  (a new cell whose first line is `# YOUR CODE` works too).
* Run the `check(...)` cell after your code: it tests your functions and prints
  PASS or FAIL for each test.
* Answer each question in the cell right below it, in one or two sentences,
  replacing *TODO*.
* Save often (Ctrl+S). Before you submit, run everything from a clean start:
  restart the kernel and run all cells, or run `python run_notebooks.py`.

**Name:** Gurijala Yatheendra Reddy

**Roll number:** 26210053

In [ ]:
# PROVIDED: run this cell first
%matplotlib inline
import numpy as np
from provided.notebook import check
from provided.task1.steps import check_constraint, load_task_frame, show_features, show_gradients

frame = load_task_frame()          # frame 276 of the descent, at quarter size

## 1.1 Spatial gradients (6 marks)

Implement central differences, without `np.gradient`:

$$
I_x(x,y)=\frac{I(x+1,y)-I(x-1,y)}{2},\qquad I_y(x,y)=\frac{I(x,y+1)-I(x,y-1)}{2}.
$$

In [ ]:
# YOUR CODE: Task 1.1

def spatial_gradients(image):
    """Central differences, equation (1.1):
        Ix(x, y) = (I(x+1, y) - I(x-1, y)) / 2,   Iy(x, y) = (I(x, y+1) - I(x, y-1)) / 2

    Returns (ix, iy): float arrays shaped like image, 0 on the one-pixel border.
    """
    image = np.asarray(image, dtype=np.float64)
    ix = np.zeros_like(image)
    iy = np.zeros_like(image)
    # x runs along the columns, so the left/right neighbours are column shifts
    ix[1:-1, 1:-1] = (image[1:-1, 2:] - image[1:-1, :-2]) / 2.0
    # y runs along the rows, so the up/down neighbours are row shifts
    iy[1:-1, 1:-1] = (image[2:, 1:-1] - image[:-2, 1:-1]) / 2.0
    return ix, iy

In [ ]:
check("1.1")

In [ ]:
grad = show_gradients(frame, spatial_gradients)

**(a)** Where are the gradients strongest, and where are they weakest?

They are strongest on crater rims, small rocks and sharp light to dark edges, like the bright crater in the lower left, because brightness changes a lot between neighbouring pixels there. They are weakest on the smooth ground between craters, where Ix and Iy are almost zero.

## 1.2 Temporal change and motion (5 marks)

Implement $I_t = I_1 - I_0$. For a small motion $(u, v)$, brightness constancy
gives the motion constraint

$$
I_x u + I_y v + I_t \approx 0.
$$

The provided cell checks it on frame 276 shifted by a known $(u, v) = (0.3, -0.2)$.

In [ ]:
# YOUR CODE: Task 1.2

def temporal_difference(image0, image1):
    """It = I1 - I0 (Task 1.2): how much each pixel's brightness changed.

    Returns a float array shaped like image0.
    """
    # cast first so that integer images cannot wrap around when subtracted
    return np.asarray(image1, dtype=np.float64) - np.asarray(image0, dtype=np.float64)

In [ ]:
check("1.2")

In [ ]:
stats = check_constraint(frame, spatial_gradients, temporal_difference)

**(a)** What mean residual does the check report (saved in `constraint_check.json`)?

The mean |residual| is about **0.0042** (max 0.106) for the shift (0.3, -0.2). That is very small compared with gradients of around 0.2, so the motion equation works well for small shifts.

**(b)** Why can one pixel not give both $u$ and $v$?

One pixel gives just one equation, Ix u + Iy v = -It, but we have two unknowns, so many (u, v) pairs fit it. From one pixel we can only find the motion along the gradient direction (the aperture problem).

## 1.3 Choosing good points to track (9 marks)

For each local window build $G = A^TA$, with one row $[I_x, I_y]$ of $A$ per
pixel, and use the smaller eigenvalue as a trackability score:

$$
G = A^TA, \qquad q(x,y)=\min(\lambda_1,\lambda_2).
$$

Select strong points with a minimum separation. The same matrix $G$ is what
Lucas-Kanade solves in Task 2.

In [ ]:
# YOUR CODE: Task 1.3

def structure_tensor(ix, iy, x, y, window_radius):
    """G = A^T A for the window centred on pixel (x, y), equation (1.3).

    Returns the 2x2 array [[sum Ix*Ix, sum Ix*Iy], [sum Ix*Iy, sum Iy*Iy]].
    """
    r = int(window_radius)
    x, y = int(round(x)), int(round(y))
    wx = ix[y - r:y + r + 1, x - r:x + r + 1]
    wy = iy[y - r:y + r + 1, x - r:x + r + 1]
    sxx = np.sum(wx * wx)
    sxy = np.sum(wx * wy)
    syy = np.sum(wy * wy)
    return np.array([[sxx, sxy], [sxy, syy]], dtype=np.float64)


def feature_score(G):
    """Trackability q = min(lambda1, lambda2) of the 2x2 matrix G, equation (1.3)."""
    # eigvalsh gives the eigenvalues of a symmetric matrix in ascending order
    return float(np.linalg.eigvalsh(G)[0])


def _window_sums(a, r):
    """Sum of a over every (2r+1) x (2r+1) window that fits inside the image.

    Uses an integral image, so it costs the same for any window size. Entry
    [i, j] of the result is the window centred on pixel (x = j + r, y = i + r).
    """
    k = 2 * r + 1
    c = np.zeros((a.shape[0] + 1, a.shape[1] + 1))
    c[1:, 1:] = np.cumsum(np.cumsum(a, axis=0), axis=1)
    return c[k:, k:] - c[:-k, k:] - c[k:, :-k] + c[:-k, :-k]


def select_features(image, max_points=200, window_radius=3, min_distance=10, threshold=None):
    """Choose up to max_points strong, well-separated points in an image.

    Returns an (N, 2) float array of [x, y], strongest first, N <= max_points.

    Same score as feature_score(structure_tensor(...)) at every pixel, but all
    windows are scored at once with integral images and the closed form of the
    smaller eigenvalue of [[a, b], [b, c]], so a full frame takes milliseconds.
    """
    ix, iy = spatial_gradients(image)
    r = int(window_radius)
    h, w = ix.shape
    if h < 2 * r + 1 or w < 2 * r + 1:
        return np.zeros((0, 2))

    a = _window_sums(ix * ix, r)
    b = _window_sums(ix * iy, r)
    c = _window_sums(iy * iy, r)
    q = (a + c) / 2.0 - np.sqrt(((a - c) / 2.0) ** 2 + b ** 2)

    ys, xs = np.mgrid[r:h - r, r:w - r]
    scores, xs, ys = q.ravel(), xs.ravel(), ys.ravel()

    # flat areas give q = 0 (up to rounding), so they are dropped
    keep = scores >= threshold if threshold is not None else scores > 1e-12
    scores, xs, ys = scores[keep], xs[keep], ys[keep]
    order = np.argsort(-scores, kind="stable")

    chosen = []
    for k in order:
        p = np.array([xs[k], ys[k]], dtype=np.float64)
        if chosen and np.min(np.linalg.norm(np.array(chosen) - p, axis=1)) < min_distance:
            continue
        chosen.append(p)
        if len(chosen) >= max_points:
            break
    return np.array(chosen, dtype=np.float64).reshape(-1, 2)

In [ ]:
check("1.3")

In [ ]:
points = show_features(frame, select_features)

**(a)** What does the minimum distance between selected points prevent?

It stops many points from bunching up on the same strong feature, since one crater rim has lots of high scoring pixels next to each other. Keeping them 10 px apart spreads the points across the whole image, which gives a better motion fit later.

**(b)** What do the eigenvalues of $G$ look like on a flat patch, an edge and a corner?

Flat area: both eigenvalues are close to 0. Edge: one is large and the other is close to 0, since brightness only changes across the edge. Corner: both are large.

**(c)** Why does a large $q$ mean a point is easy to track?

q is the smaller eigenvalue of G, the matrix Lucas-Kanade has to invert. If q is large, the patch has texture in both directions, so both u and v can be found reliably and small noise does not change the answer much.